In [7]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer

In [8]:
lr = LinearRegression()


ames = pd.read_csv("AmesHousing.csv")
X = ames[["Gr Liv Area", "TotRms AbvGrd"]]
y = ames["SalePrice"]



X_train, X_test, y_train, y_test = train_test_split(X, y)

X_train_s = (X_train - X_train.mean())/X_train.std()

lr_fitted = lr.fit(X_train_s, y_train)
lr_fitted.coef_

array([ 70935.17656886, -16825.68984943])

In [9]:
X_test_s = (X_test - X_train.mean())/X_train.std()
y_preds = lr_fitted.predict(X_test_s)

r2_score(y_test, y_preds)

0.5130046146484101

In [10]:
new_house = pd.DataFrame(data = {"Gr Liv Area": [889], "TotRms AbvGrd": [6]})
new_house

,Gr Liv Area,TotRms AbvGrd
0,889,6


In [11]:
new_house_s = (new_house - X_train.mean())/X_train.std()
lr_fitted.predict(new_house_s)

array([102116.81994397])

Creating a Pipeline Objects

In [12]:
lr = LinearRegression()

lr_pipeline = Pipeline(
  [StandardScaler(),lr]
)

lr_pipeline

TypeError: 'LinearRegression' object is not subscriptable

In [13]:
lr_pipeline = Pipeline(
  [("standardize", StandardScaler()),
  ("linear_regression", LinearRegression())]
)

lr_pipeline

Pipeline(steps=[('standardize', StandardScaler()),
                ('linear_regression', LinearRegression())])

In [14]:
lr_pipeline_fitted = lr_pipeline.fit(X_train, y_train)

y_preds = lr_pipeline_fitted.predict(X_test)
r2_score(y_test, y_preds)

0.5130046146484101

In [15]:
lr_pipeline_fitted.predict(new_house)

array([102116.81994397])

In [16]:
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)


lr_pipeline = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
)

lr_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('dummify',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['Bldg Type']),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area',
                                                   'TotRms AbvGrd'])])),
                ('linear_regression', LinearRegression())])

In [17]:
X = ames.drop("SalePrice", axis = 1)
y = ames["SalePrice"]



X_train, X_test, y_train, y_test = train_test_split(X, y)

lr_fitted = lr_pipeline.fit(X_train, y_train)

In [18]:
ct_fitted = ct.fit(X_train)

ct.transform(X_train)

array([[ 1.        ,  0.        ,  0.        , ...,  0.        , -1.13046514,
        -1.56704142],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.56639801,
         0.35415836],
       [ 0.        ,  0.        ,  0.        , ...,  1.        , -0.61676114,
        -0.92664149],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.24810298,
        -0.28624157],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  0.61008488,
         0.35415836],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,  0.98478662,
         1.6349582 ]])

In [19]:
ct.transform(X_test)

array([[ 1.        ,  0.        ,  0.        , ...,  0.        , -0.90080924,
        -0.28624157],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.01643255,
         0.99455828],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.90080924,
        -0.28624157],
       ...,
       [ 0.        ,  1.        ,  0.        , ...,  0.        , -0.81015559,
        -0.28624157],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -0.30853874,
        -0.28624157],
       [ 1.        ,  0.        ,  0.        , ...,  0.        , -1.0639858 ,
        -0.92664149]])

In [20]:
lr_pipeline_fitted.named_steps['linear_regression'].coef_

array([ 70919.03108633, -16821.86017148])

In [21]:
lr_pipeline = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")


ct.fit_transform(X_train)

,dummify__Bldg Type_1Fam,dummify__Bldg Type_2fmCon,dummify__Bldg Type_Duplex,dummify__Bldg Type_Twnhs,dummify__Bldg Type_TwnhsE,standardize__Gr Liv Area,standardize__TotRms AbvGrd
156,1.0,0.0,0.0,0.0,0.0,-1.130465,-1.567041
2560,1.0,0.0,0.0,0.0,0.0,-0.566398,0.354158
299,0.0,0.0,0.0,0.0,1.0,-0.616761,-0.926641
961,1.0,0.0,0.0,0.0,0.0,-0.062767,0.354158
1518,1.0,0.0,0.0,0.0,0.0,0.396545,0.354158
...,...,...,...,...,...,...,...
1886,1.0,0.0,0.0,0.0,0.0,-1.239250,-0.926641
1921,1.0,0.0,0.0,0.0,0.0,-0.888722,-0.286242
1534,1.0,0.0,0.0,0.0,0.0,-0.248103,-0.286242
2899,1.0,0.0,0.0,0.0,0.0,0.610085,0.354158


In [22]:
ct_inter = ColumnTransformer(
  [
    ("interaction", PolynomialFeatures(interaction_only = True), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
).set_output(transform = "pandas")

ct_inter.fit_transform(X_train)

,interaction__1,interaction__Gr Liv Area,interaction__TotRms AbvGrd,interaction__Gr Liv Area TotRms AbvGrd
156,1.0,936.0,4.0,3744.0
2560,1.0,1216.0,7.0,8512.0
299,1.0,1191.0,5.0,5955.0
961,1.0,1466.0,7.0,10262.0
1518,1.0,1694.0,7.0,11858.0
...,...,...,...,...
1886,1.0,882.0,5.0,4410.0
1921,1.0,1056.0,6.0,6336.0
1534,1.0,1374.0,6.0,8244.0
2899,1.0,1800.0,7.0,12600.0


In [23]:
ct_dummies = ColumnTransformer(
  [("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"])],
  remainder = "passthrough"
).set_output(transform = "pandas")

ct_inter = ColumnTransformer(
  [
    ("interaction", PolynomialFeatures(interaction_only = True), ["remainder__TotRms AbvGrd", "dummify__Bldg Type_1Fam"]),
  ],
  remainder = "drop"
).set_output(transform = "pandas")

X_train_dummified = ct_dummies.fit_transform(X_train)
X_train_dummified

,dummify__Bldg Type_1Fam,dummify__Bldg Type_2fmCon,dummify__Bldg Type_Duplex,dummify__Bldg Type_Twnhs,dummify__Bldg Type_TwnhsE,remainder__Order,remainder__PID,remainder__MS SubClass,remainder__MS Zoning,remainder__Lot Frontage,remainder__Lot Area,remainder__Street,remainder__Alley,remainder__Lot Shape,remainder__Land Contour,remainder__Utilities,remainder__Lot Config,remainder__Land Slope,remainder__Neighborhood,remainder__Condition 1,remainder__Condition 2,remainder__House Style,remainder__Overall Qual,remainder__Overall Cond,remainder__Year Built,remainder__Year Remod/Add,remainder__Roof Style,remainder__Roof Matl,remainder__Exterior 1st,remainder__Exterior 2nd,remainder__Mas Vnr Type,remainder__Mas Vnr Area,remainder__Exter Qual,remainder__Exter Cond,remainder__Foundation,remainder__Bsmt Qual,remainder__Bsmt Cond,remainder__Bsmt Exposure,remainder__BsmtFin Type 1,remainder__BsmtFin SF 1,...,remainder__Heating QC,remainder__Central Air,remainder__Electrical,remainder__1st Flr SF,remainder__2nd Flr SF,remainder__Low Qual Fin SF,remainder__Gr Liv Area,remainder__Bsmt Full Bath,remainder__Bsmt Half Bath,remainder__Full Bath,remainder__Half Bath,remainder__Bedroom AbvGr,remainder__Kitchen AbvGr,remainder__Kitchen Qual,remainder__TotRms AbvGrd,remainder__Functional,remainder__Fireplaces,remainder__Fireplace Qu,remainder__Garage Type,remainder__Garage Yr Blt,remainder__Garage Finish,remainder__Garage Cars,remainder__Garage Area,remainder__Garage Qual,remainder__Garage Cond,remainder__Paved Drive,remainder__Wood Deck SF,remainder__Open Porch SF,remainder__Enclosed Porch,remainder__3Ssn Porch,remainder__Screen Porch,remainder__Pool Area,remainder__Pool QC,remainder__Fence,remainder__Misc Feature,remainder__Misc Val,remainder__Mo Sold,remainder__Yr Sold,remainder__Sale Type,remainder__Sale Condition
156,1.0,0.0,0.0,0.0,0.0,157,535350040,20,RL,74.0,5868,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Story,5,7,1956,2000,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,BLQ,248.0,...,Ex,Y,SBrkr,936,0,0,936,1.0,0.0,1,0,2,1,TA,4,Typ,0,NaN,Attchd,1956.0,Fin,1.0,308.0,TA,TA,Y,0,0,80,0,160,0,NaN,NaN,NaN,0,5,2010,WD,Normal
2560,1.0,0.0,0.0,0.0,0.0,2561,534476100,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Story,5,5,1951,1951,Gable,CompShg,HdBoard,HdBoard,Stone,144.0,TA,TA,CBlock,TA,TA,No,ALQ,996.0,...,Ex,Y,FuseA,1216,0,0,1216,1.0,0.0,1,0,3,1,TA,7,Typ,0,NaN,Attchd,1951.0,RFn,1.0,280.0,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,5,2006,WD,Normal
299,0.0,0.0,0.0,0.0,1.0,300,909455040,120,RM,35.0,3907,Pave,NaN,IR1,HLS,AllPub,Inside,Mod,Blueste,Norm,Norm,1Story,8,5,1989,1989,Gable,CompShg,HdBoard,HdBoard,NaN,0.0,Gd,TA,CBlock,Gd,TA,Av,GLQ,76.0,...,Gd,Y,SBrkr,1191,0,0,1191,0.0,0.0,2,0,2,1,Gd,5,Typ,1,TA,Attchd,1989.0,Unf,2.0,531.0,TA,TA,Y,112,81,0,0,0,0,NaN,NaN,NaN,0,3,2010,WD,Normal
961,1.0,0.0,0.0,0.0,0.0,962,916386140,20,RL,73.0,8925,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,Timber,Norm,Norm,1Story,8,5,2007,2007,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,Gd,TA,PConc,Gd,TA,Av,GLQ,16.0,...,Ex,Y,SBrkr,1466,0,0,1466,0.0,0.0,2,0,3,1,Gd,7,Typ,0,NaN,Attchd,2007.0,Fin,3.0,610.0,TA,TA,Y,100,18,0,0,0,0,NaN,NaN,NaN,0,7,2009,WD,Normal
1518,1.0,0.0,0.0,0.0,0.0,1519,909175100,50,RL,60.0,8400,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,SWISU,Norm,Norm,1.5Fin,6,5,1925,1950,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,PConc,TA,TA,No,Rec,423.0,...,Fa,Y,SBrkr,1390,304,0,1694,0.0,0.0,2,0,4,1,TA,7,Typ,1,Gd,Detchd,1925.0,Unf,2.0,576.0,TA,TA,Y,342,0,128,0,0,0,NaN,NaN,NaN,0,6,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1886,1.0,0.0,0.0,0.0,0.0,1887,534276290,20,RL,NaN,8339,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Story,5,7,1959,1959,Gable,CompShg,MetalSd,Meta

In [24]:
ct_inter.fit_transform(X_train_dummified)

,interaction__1,interaction__remainder__TotRms AbvGrd,interaction__dummify__Bldg Type_1Fam,interaction__remainder__TotRms AbvGrd dummify__Bldg Type_1Fam
156,1.0,4.0,1.0,4.0
2560,1.0,7.0,1.0,7.0
299,1.0,5.0,0.0,0.0
961,1.0,7.0,1.0,7.0
1518,1.0,7.0,1.0,7.0
...,...,...,...,...
1886,1.0,5.0,1.0,5.0
1921,1.0,6.0,1.0,6.0
1534,1.0,6.0,1.0,6.0
2899,1.0,7.0,1.0,7.0


## Practice Activity

In [25]:
# Using only the size and number of rooms.

X = ames[["Gr Liv Area", "TotRms AbvGrd"]]
y = ames["SalePrice"]

lr_pipeline = Pipeline(
  [("linear_regression", LinearRegression())]
)

'''
lr_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y)

lr_pipeline_fitted = lr_pipeline.fit(X_train,y_train)

y_pred = lr_pipeline_fitted.predict(X_test)

mean_squared_error(y_test, y_pred)
'''

scores = cross_val_score(lr_pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(abs(scores.mean()))

NameError: name 'cross_val_score' is not defined

In [ ]:
# Using size, number of rooms, and building type.

X = ames[["Gr Liv Area", "TotRms AbvGrd", "Bldg Type"]]
y = ames["SalePrice"]

ct = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)

lr_pipeline2 = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
)

'''
lr_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y)

lr_pipeline_fitted = lr_pipeline.fit(X_train,y_train)

y_pred = lr_pipeline_fitted.predict(X_test)

mean_squared_error(y_test, y_pred)
'''

scores = cross_val_score(lr_pipeline2, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(abs(scores.mean()))



In [26]:
# Using size and building type, and their interaction

X = ames[["Gr Liv Area", "Bldg Type"]]
y = ames["SalePrice"]

ct_dummies = ColumnTransformer(
  [("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
  ("standardize", StandardScaler(), ["Gr Liv Area"])], 
  remainder = "drop"
).set_output(transform = "pandas")

ct_inter = ColumnTransformer(
  [
    ("interaction1", PolynomialFeatures(interaction_only = True), ["standardize__Gr Liv Area", "dummify__Bldg Type_1Fam"]), 
    ("interaction2", PolynomialFeatures(interaction_only = True), ["standardize__Gr Liv Area", "dummify__Bldg Type_2fmCon"]),
    ("interaction3", PolynomialFeatures(interaction_only = True), ["standardize__Gr Liv Area", "dummify__Bldg Type_Duplex"]),
    ("interaction4", PolynomialFeatures(interaction_only = True), ["standardize__Gr Liv Area", "dummify__Bldg Type_Twnhs"]),
    ("interaction5", PolynomialFeatures(interaction_only = True), ["standardize__Gr Liv Area", "dummify__Bldg Type_TwnhsE"])
  ],
  remainder = "passthrough"
).set_output(transform = "pandas")

lr_pipeline3 = Pipeline(
  [("preprocessing", ct_dummies),
  ("interaction", ct_inter),
  ("linear_regression", LinearRegression())]
)

scores = cross_val_score(lr_pipeline3, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(abs(scores.mean()))


NameError: name 'cross_val_score' is not defined

In [ ]:
# Using a 5-degree polynomial on size, a 5-degree polynomial on number of rooms, and also building type

X = ames[["Gr Liv Area", "Bldg Type", "TotRms AbvGrd"]]
y = ames["SalePrice"]

ct_dummies = ColumnTransformer(
  [("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
  ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"])], 
  remainder = "drop"
).set_output(transform = "pandas")

ct_poly = ColumnTransformer(
  [
    ("poly1", PolynomialFeatures(degree = 5, interaction_only = False), ["standardize__Gr Liv Area"]), 
    ("poly2", PolynomialFeatures(degree = 5, interaction_only = False), ["standardize__TotRms AbvGrd"])
  ],
  remainder = "passthrough"
).set_output(transform = "pandas")

lr_pipeline4 = Pipeline(
  [("preprocessing", ct_dummies),
  ("preprocessing2", ct_poly),
  ("linear_regression", LinearRegression())]
)

scores = cross_val_score(lr_pipeline4, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(abs(scores.mean()))


# Cross-Validating

In [ ]:
from sklearn.model_selection import cross_val_score

X = ames.drop("SalePrice", axis = 1)
y = ames["SalePrice"]


ct = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)

lr_pipeline_1 = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")


scores = cross_val_score(lr_pipeline_1, X, y, cv=5, scoring='r2')
scores

In [28]:
scores.mean()

NameError: name 'scores' is not defined

Cross validated r-squared is 0.53311

## Tuning

In [59]:
from sklearn.model_selection import GridSearchCV

X = ames.drop("SalePrice", axis = 1)
y = ames["SalePrice"]

ct_poly = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("polynomial", PolynomialFeatures(), ["Gr Liv Area"])
  ],
  remainder = "drop"
)

lr_pipeline_poly = Pipeline(
  [("preprocessing", ct_poly),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")

degrees = {'preprocessing__polynomial__degree': np.arange(1, 10)}

gscv = GridSearchCV(lr_pipeline_poly, degrees, cv = 5, scoring='r2')

In [29]:
gscv_fitted = gscv.fit(X, y)

gscv_fitted.cv_results_

NameError: name 'gscv' is not defined

In [ ]:
gscv_fitted.cv_results_['mean_test_score']

In [30]:
pd.DataFrame(data = {"degrees": np.arange(1, 10), "scores": gscv_fitted.cv_results_['mean_test_score']})

NameError: name 'gscv_fitted' is not defined

In [62]:
# Consider one hundred modeling options for house price: 
# - House size, trying degrees 1 through 10
# - Number of rooms, trying degrees 1 through 10
# - Building Type

X = ames.drop("SalePrice", axis=1)
y = ames["SalePrice"]

# Define the ColumnTransformer with PolynomialFeatures and OneHotEncoder
ct_poly = ColumnTransformer(
    transformers=[
        ("dummify", OneHotEncoder(sparse_output=False), ["Bldg Type"]),
        ("polynomial1", PolynomialFeatures(), ["Gr Liv Area"]),
        ("polynomial2", PolynomialFeatures(), ["TotRms AbvGrd"])
    ],
    remainder="drop"
)

lr_pipeline_poly = Pipeline([
    ("preprocessing", ct_poly),
    ("linear_regression", LinearRegression())
]).set_output(transform="pandas")

param_grid = {
    'preprocessing__polynomial1__degree': np.arange(1, 11),
    'preprocessing__polynomial2__degree': np.arange(1, 11)
}

# Set up GridSearchCV with the updated param_grid
gscv = GridSearchCV(lr_pipeline_poly, param_grid, cv=5, scoring='r2')

In [31]:
gscv_fitted = gscv.fit(X, y)

gscv_fitted.cv_results_

NameError: name 'gscv' is not defined

In [ ]:
gscv_fitted.cv_results_['mean_test_score']

In [32]:
degrees = pd.DataFrame(gscv_fitted.cv_results_["params"])
results = degrees.assign(scores=gscv_fitted.cv_results_['mean_test_score'])
results.sort_values(by='scores', ascending=False)

NameError: name 'gscv_fitted' is not defined

GrLivArea with degree = 3 and TotRmsAbvGrd with degree = 1 has the highest R-squared